# Setting up a D1SUNED study

**~30 min.**

In this training we set up a complete **D1SUNED shutdown dose rate (SDDR)** study with F4Enix,
starting from the CAD-derived MCNP model developed in GEOUNED training session.
The workflow follows the F4Enix tutorial
*Setting up a D1SUNED study* almost step by step, applied to our model, and adds an **atlas** of
SDDR Radmaps at several cooling times, obtained by rescaling a single mesh tally.

**The D1SUNED idea in short.** In D1SUNED, a single mode N P coupled neutron-photon run with
additional files is employed to simulate decay photons:

* the **reaction file** (`react`) lists the reactions to consider, *parent -(MT)-> daughter*;
* the **irradiation file** (`irrad`) gives, for each daughter, its decay constant and the
  **time correction factor** (TCF), which accounts for the irradiation history and the decay.

The result is the SDDR at a given cooling time. If tallies are **binned by daughter** nuclide, each
contribution can be rescaled afterwards to other cooling times, which is how we build the atlas.

In [ ]:
# Run this cell first. It sets up the paths used by the whole notebook.
from pathlib import Path

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

%matplotlib inline
print("data folder :", DATA.resolve())
print("output folder:", OUT.resolve())

## Introduction to the problem

`example.mcnp` is a PFC#5 clamp region (lower clamp, tie rods, nuts, cooling pipes) converted
from CAD with GEOUNED. After the conversion:

* materials 10 (Steel), 20 (Inconel) and 30 (Water) were added;
* it is set up for a **lost particle check**: `VOID` card and a neutron source on the bounding
  sphere (surface 406) pointing inwards.

Our objective is the SDDR around the clamp (**tally 104**) and an SDDR map (**mesh tally 214**)
at 24 h, 11.6 days, 30 days and 1 year after shutdown.

| XZ section through the tie rods (y = 30 cm) | XY section (z = -700 cm) |
|:---:|:---:|
| ![XZ section y = 30 cm](geometry_plots/xz_y30_mat.png) | ![XY section z = -700 cm](geometry_plots/xy_z-700_mat.png) |

Whole model with cell numbers: cell 88, between the bounding box of the model and the sphere, is the cell of tally 104.

![XZ section with cell numbers](geometry_plots/xz_y7_cells.png)


In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keep the training output readable

from f4enix.input.MCNPinput import D1S_Input

inp = D1S_Input.from_input(DATA / "d1suned/example.mcnp")

# Cells grouped by material and density
inp.get_cells_summary().groupby(["material", "density"]).size()

## Preparation of the model

### Materials

The cells use three materials:

| Material | Density [g/cm³] | Composition |
|---|---|---|
| 10 | 7.93 | SS316L(N)-IG (clamp, nuts, pads) |
| 20 | 5.5 | Inconel 718 (tie rods, studs) |
| 30 | 1.0 | water (cooling pipes) |


### Source and run settings

The lost particle check source is kept: it is an inward cosine source on the sphere 406 that
irradiates the clamp uniformly from all directions. Its weight is $W = \pi R^2$, so that
**one source particle = 1 n/cm²** of isotropic fluence inside the sphere. As a consequence, the
intensities of the irradiation scenario below are **neutron fluxes [n/cm²/s]**.

We only need to remove the `VOID` card and give the neutrons an energy spectrum. Since the decay
pathways are taken below for the *500 MW port cell* spectrum, the source has a **port-cell-like
shape**: a strong thermal peak, a slowing-down region and a small residual 14 MeV D-T component.

| Energy range | Fraction of the neutrons |
|---|---|
| < 1 eV | 35 % |
| 1 eV - 0.1 MeV | 29 % |
| 0.1 - 1 MeV | 20 % |
| 1 - 10 MeV | 14 % |
| > 10 MeV | 2 % |

> **Note:** this is an illustrative spectrum for the training, hard-coded by hand. It is not the
> group-wise spectrum used to build the pathway library, which is not distributed with F4Enix.

In [ ]:
del inp.other_data["VOID"]
inp.other_data["SDEF"] = "SDEF PAR=N NRM=-1 SUR=406 WGT=6.0899732e+04 DIR=d1 ERG=d2\n"
# Illustrative port-cell-like spectrum: energy bin boundaries [MeV]
inp.other_data["SI2"] = (
    "SI2 H 1e-11 1e-8 1e-7 1e-6 1e-5 1e-4 1e-3 1e-2\n"
    "      0.1 0.5 1 2 5 10 13.5 14.5\n"
)
# Probability of each bin (the first entry of a histogram is always 0)
inp.other_data["SP2"] = (
    "SP2 D 0 0.03 0.27 0.05 0.05 0.05 0.05 0.06\n"
    "      0.08 0.12 0.08 0.06 0.05 0.03 0.01 0.01\n"
)

inp.other_data["PHYS:P"] = "PHYS:P J 1\n"
inp.other_data["PRDMP"] = "PRDMP 2J 1\n"  # write the mctal file
inp.other_data["LOST"] = "LOST 100\n"  # a few lost particles are tolerated
inp.other_data["NPS"] = "NPS 1e7\n"  # a few lost particles are tolerated
inp.other_data["MODE"] = "MODE N P\n"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Read the spectrum back from the source cards
edges = np.array(str(inp.other_data["SI2"]).split()[2:], dtype=float)  # [MeV]
probabilities = np.array(str(inp.other_data["SP2"]).split()[3:], dtype=float)  # skip "SP2 D 0"

# Per unit lethargy, so that equal areas on the log axis mean equal numbers of neutrons
per_lethargy = probabilities / np.log(edges[1:] / edges[:-1])

plt.figure(figsize=(10, 5))
plt.stairs(per_lethargy, edges, color="tab:blue", linewidth=2)
plt.xscale("log")
plt.xlabel("Neutron energy [MeV]")
plt.ylabel("Source probability per unit lethargy [-]")
plt.title("Illustrative port-cell-like source spectrum")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

## Irradiation scenario and identification of decay pathways

First thing to do is to define the irradiation scenario(s) to be analyzed. This can be read from a
FISPACT input or a d1stime legacy file if available, or it can be directly created in F4Enix.
As in the F4Enix tutorial, we use a dummy scenario.

In [ ]:
from f4enix.core.irradiation import IrradiationScenario, Pulse
from f4enix.core.constants import TIME_UNITS

NORM = 1e10  # reference neutron flux [n/cm2/s]

# Dummy irradiation scenario: 2 cycles of 1 day ON, 2 days OFF, then 1 year OFF
# then again 2 cycles of 1 day ON, 2 days OFF
pulses = (
    [Pulse(1, NORM, TIME_UNITS.DAY), Pulse(2, 0, TIME_UNITS.DAY)] * 2
    + [Pulse(1, 0, TIME_UNITS.YEAR)]
    + [Pulse(1, NORM, TIME_UNITS.DAY), Pulse(2, 0, TIME_UNITS.DAY)] * 2
)
irr_scenario = IrradiationScenario(pulses)

Next, we need to identify which nuclides should be tracked during the D1SUNED calculation.
Usually, a previous activation calculation (e.g. FISPACT) is run to identify the main decay pathways
contributing to the SDDR. Here we use the **decay pathway library of F4Enix**, assuming that the
spectrum at the clamp is similar to the *500 MW port cell* one and that our scenario is similar
to *DT2*.

> **Important:** the pathways must cover **all the cooling times of interest**. A nuclide that is
> negligible at 24 h (e.g. Co-60) can dominate at 1 year. Since we want the atlas at 24 h, 11.6 d,
> 30 d and 1 y, all four cooling times are requested.

In [ ]:
from f4enix.output.decay_pathways import PathwayLibrary, AVAIL_IRR_SCENARIO, AVAIL_SPECTRUM
from f4enix.core.material_library import SS316LNIG, INCONEL718

plib = PathwayLibrary()
df = plib.get_pathways(
    materials=[SS316LNIG, INCONEL718],  # the materials of our model
    cooling_times=["24 h", "11.6 d", "30 d", "1 y"],
    spectrum=[AVAIL_SPECTRUM.PortCell500MW],
    irradiation_scenario=[AVAIL_IRR_SCENARIO.DT2],
    dose=95,  # interested in getting 95% of the dose
)

df

## Irradiation and Reaction files production

### IRRAD file

First of all, let's extract the radioactive daughter isotopes for which the time correction
factors need to be computed.

In [ ]:
daughters = df.reset_index()[["element", "isotope"]].drop_duplicates()
daughters

Then the time correction factors are computed **at shutdown** and the irradiation file is created.
`norm` is the reference intensity: the TCFs are computed per unit of `NORM`, so the tallies will
have to be multiplied by `NORM` (see the tallies section).

In [ ]:
from f4enix.input.d1suned import IrradiationFile
from f4enix.core.irradiation import Nuclide

nuclides = []
for element, isotope in daughters.values:
    nuclides.append(Nuclide.from_formula(f"{element}{isotope}"))

irr_file = IrradiationFile.from_irradiation_schedules(
    nuclides,
    [irr_scenario],
    norm=NORM,  # normalization factor
)

inp.irrad_file = irr_file  # assign the irrad file to the input
irr_file.to_df()

### REAC file

The fastest way to produce a reac file is the `get_reaction_file()` method of `D1S_Input`.
Given a D1S library (here `99c`), it scans the materials of the input and adds every available
reaction that leads from an isotope of the materials to one of the daughters of the irrad file.

In [ ]:
from f4enix.input.libmanager import LibManager

lm = LibManager()

inp.get_reaction_file(lm, "99c")
inp.reac_file

This list is longer than the one of the decay pathway library, so we can refine it. First, the
pathway strings are converted into `Pathway` objects

In [ ]:
from f4enix.output.fispact_legacy_out import Pathway

pathways = []
for string in df.reset_index()["pathway"].unique():
    pathways.append(Pathway.from_string(string))

pathways

and then the reactions that are not needed are filtered out. Daughters of the irrad file that are no
longer produced by any reaction are removed too, keeping the two files consistent.

In [ ]:
inp.filter_by_paths(pathways)
inp.reac_file

## D1SUNED input file finalization

Now that the irrad and reac files are ready, the input needs some final touches. First, the D1S
library is assigned to the parent isotopes of the reactions, and the transport library
(FENDL-3.2c, `32c`) to all the other isotopes. PIKMT card is added automatically.

In [ ]:
inp.smart_translate(
    "99c",  # activation library to use for parents
    "32c",  # transport library to use for all other isotopes
    lm,  # LibManager of choice (default one can be used in most cases)
)

print(inp.mat_section["M10"].to_text())
print(inp.mat_section["M20"].to_text())
print(inp.mat_section["M30"].to_text())
print(inp.other_data["PIKMT"])

### Tallies

Both tallies give the ambient dose equivalent rate H\*(10) of the decay photons (ICRP-74
flux-to-dose coefficients, `DE`/`DF` cards, in pSv·cm²) and are **binned by daughter**:

* **F104**: average SDDR in the void around the clamp (cell 88, between the bounding box of the
  model and the sphere). MCNP cannot compute the volume of this cell, so it is given with `SD`.
  The daughter bins are added with `add_daughter_contribution_from_irr()` (`FU` card).
* **FMESH214**: SDDR map on 5 cm voxels around the clamp. On a mesh the daughter binning is
  done with `etag=nuc` and the list of daughters in the `emesh` keyword.

The multiplier `FM` converts the result to µSv/h and restores the normalization of the TCFs:
$FM = NORM \times 3600 \times 10^{-6}$ (pSv/s → µSv/h).

In [ ]:
import numpy as np

FM = f"{NORM * 3600 * 1e-6:.4e}"  # [uSv/h]
DE = (
    "0.01 0.015 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.10 0.15 0.20\n"
    "      0.3 0.4 0.5 0.6 0.8 1.0 2.0 4.0 6.0 8.0 10.0\n"
)
DF = (
    "0.0485 0.1254 0.2050 0.2999 0.3381 0.3572 0.3780 0.4066 0.4399 0.5172\n"
    "      0.7523 1.0041 1.5083 1.9958 2.4657 2.9082 3.7269 4.4834 7.4896 12.0153\n"
    "      15.9873 19.9191 23.7600\n"
)

# --- F104: SDDR in cell 88 (sphere 406 minus the bounding box of the model) ---
radius = 139.22998
box = (936.685 - 761.3) * (65.185 + 51.0) * (-600.96688 + 774.95)
volume = 4 / 3 * np.pi * radius**3 - box

inp.add_F_tally(104, ["P"], [88], description="SDDR around the clamp [uSv/h]", multiplier=FM, add_SD=False)
inp.other_data["SD104"] = f"SD104 {volume:.5e}\n"
inp.other_data["DE104"] = "DE104 " + DE
inp.other_data["DF104"] = "DF104 " + DF
inp.add_daughter_contribution_from_irr("F104")  # FU card with the daughters

# --- FMESH214: SDDR map, binned by daughter (etag=nuc) ---
daughter_ids = sorted(int(d.write_to_int_string()) for d in inp.irrad_file.get_daughters())
inp.other_data["FC214"] = "FC214 SDDR mesh [uSv/h]\n"
inp.other_data["FMESH214"] = (
    "FMESH214:P GEOM=rec ORIGIN= 755 -55 -780 OUT=ij\n"
    "      IMESH= 945  IINTS= 38\n"
    "      JMESH= 70   JINTS= 25\n"
    "      KMESH= -595 KINTS= 37\n"
    "      etag=nuc emesh= " + " ".join(str(d) for d in daughter_ids) + "\n"
)
inp.other_data["DE214"] = "DE214 " + DE
inp.other_data["DF214"] = "DF214 " + DF
inp.other_data["FM214"] = f"FM214 {FM}\n"

print(inp.other_data["F104"])
print(inp.other_data["FMESH214"])

Last, the `FILES` card tells D1SUNED where to find the reac (unit 21) and irrad (unit 22) files.
The input and the two auxiliary files are written in the `d1s_run` folder.

In [ ]:
from pathlib import Path

inp.other_data["FILES"] = "FILES 21 react 3J\n      22 irrad\n"

RUN = OUT / "d1s_run"
RUN.mkdir(exist_ok=True)
inp.write(RUN / "d1s_example.mcnp")
inp.irrad_file.write(RUN)  # -> d1s_run/irrad
inp.reac_file.write(RUN)  # -> d1s_run/react
sorted(p.name for p in RUN.iterdir())

## Post-processing of results

### Daughter contributions to the dose tally

The tally is binned by daughter, so we can look at the contribution of each radionuclide to the
SDDR (at shutdown, since the TCFs were computed at shutdown).

In [ ]:
from f4enix.output.mctal import Mctal

# Read your mctal file
mctal = Mctal(str(RUN / "mctal"))

# Normalize dose tally
tallynum = 104
mctal.set_d1s_relative_contribution(tallynum)
dose_tally = mctal.tallydata[tallynum]
dose_tally

In [ ]:
import matplotlib.pyplot as plt

def nuclide_name(zaid):
    return "other" if zaid == 0 else Nuclide.from_int_string(str(zaid)).write_to_formula()


def plot_radioisotope_contribution(tally, title="Daughter contribution to dose tally"):
    plt.figure(figsize=(10, 5))
    plt.bar(tally["Daughter"].map(nuclide_name), tally["Normalized Value"], color="tab:blue")
    plt.xlabel("Daughter")
    plt.ylabel("Relative dose contribution [-]")
    plt.title(title)
    plt.grid(axis="y")
    plt.tight_layout()
    plt.show()


# Let's plot the daughters relative contributions
plot_radioisotope_contribution(dose_tally, "Daughter contribution at shutdown")

### Rescaling to other cooling times

Each daughter contribution can be moved to another cooling time $t$ by multiplying it by its decay
factor $e^{-\lambda t}$ (`get_scaling_factors_cooling_time()`). The total SDDR is then the sum over
the daughters:

$$\mathrm{SDDR}(t) = \sum_d \mathrm{SDDR}_d(0)\, e^{-\lambda_d t}$$

This is valid for the cooling times covered by the selected pathways (here 24 h to 1 y).
For a **different irradiation scenario**, `get_scaling_factors_new_scenario()` works the same way.

In [ ]:
import pandas as pd

COOLING_TIMES = [
    (24, TIME_UNITS.HOUR),
    (11.6, TIME_UNITS.DAY),
    (30, TIME_UNITS.DAY),
    (1, TIME_UNITS.YEAR),
]

# Scaling factors: one row per daughter, one column per cooling time
scaling_factors = pd.concat(
    [irr_file.get_scaling_factors_cooling_time(ct) for ct in COOLING_TIMES], axis=1
)
scaling_factors

In [ ]:
# SDDR of each daughter at each cooling time [uSv/h]
shutdown = dose_tally.set_index("Daughter")["Value"]
sddr = scaling_factors.mul(shutdown, axis=0)
sddr.index = sddr.index.map(nuclide_name)

print("Total SDDR around the clamp [uSv/h]")
print(sddr.sum().to_string())

# Relative contribution of each daughter at each cooling time
(sddr / sddr.sum() * 100).T.plot.bar(stacked=True, figsize=(10, 5), rot=0)
plt.ylabel("Contribution to the SDDR [%]")
plt.xlabel("Cooling time")
plt.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

### SDDR atlas at several cooling times

The same rescaling applies to the mesh tally, voxel by voxel. Steps:

1. read the meshtal: use from_d1s_meshtal to properly generate the daughter bins;
2. `rescale_dose_map()` rescales Daugthers contributions and total dose arrays;
3. the rescaled maps are collected in one mesh, one array per cooling time.


In [ ]:
from f4enix.input.MCNPinput import Input
from f4enix.output.meshtal.meshtal import Meshtal

run_input = Input.from_input(RUN / "d1s_example.mcnp")  # input where the FMESH is defined
meshtal = Meshtal.from_d1s_meshtal(RUN / "meshtal", run_input)
meshtal.readMesh()
fmesh = meshtal.mesh[214]

atlas_mesh = fmesh.grid.copy()
atlas_mesh.clear_data()
for label in scaling_factors.columns:
    rescaled = fmesh.rescale_dose_map(scaling_factors, label)
    # copy: the array would otherwise point to memory freed with `rescaled`
    atlas_mesh[f"SDDR {label}"] = np.array(rescaled["Value - Total_eTot"])

atlas_mesh["Rel. error (shutdown)"] = np.array(fmesh.grid["Error - Total_eTot"])

Path(OUT, "atlas").mkdir(exist_ok=True)
atlas_mesh.save(OUT / "atlas/sddr_mesh.vtr")  # to be opened in ParaView if needed
atlas_mesh

Finally, the F4Enix `MeshPlotter` cuts the mesh with planes (`slice()`: name, origin and normal
of each plane) and plots each slice with a common colour scale. The `Atlas` collects the images
in a Word document, one section per cooling time.

In [ ]:
import pyvista as pv
from f4enix.output.plotter import MeshPlotter, Atlas

pv.global_theme.allow_empty_mesh = True

stl = pv.read(DATA / "d1suned/stl_clamp.stl")

plotter = MeshPlotter(atlas_mesh, stl.scale(0.1))
slices = plotter.slice(
    [
        # [name, x, y, z, normal x, normal y, normal z]
        ["XZ plane y = 30 cm", 849, 30, -688, 0, 1, 0],
        ["XY plane z = -700 cm", 849, 7, -700, 0, 0, 1],
    ]
)

atlas = Atlas("SDDR atlas")
fig, axes = plt.subplots(2, 2, figsize=(12, 9))  # preview of the XZ slices
for label, ax in zip(scaling_factors.columns, axes.flat):
    images = plotter.plot_slices(
        slices, f"SDDR {label}", min_max=(1e1, 1e5), n_colors=8, scale_title="uSv/h"
    )
    atlas.add_section(f"SDDR {label} after shutdown", images)
    ax.imshow(images[0][1])  # images: list of (slice name, image)
    ax.set_title(f"{label} after shutdown")
    ax.axis("off")

atlas.save(OUT / "atlas")  # -> atlas/SDDR atlas.docx
plt.tight_layout()
plt.show()

## Summary

With a handful of F4Enix commands we went from a GEOUNED model to a full D1SUNED study:

1. materials and source added to the model (`Material`, `other_data`);
2. decay pathways selected from the library (`PathwayLibrary.get_pathways`);
3. irrad and reac files created and made consistent (`IrradiationFile.from_irradiation_schedules`,
   `get_reaction_file`, `filter_by_paths`);
4. libraries, `PIKMT` and daughter-binned tallies added (`smart_translate`, `add_PIKMT_card`,
   `add_daughter_contribution_from_irr`, `etag=nuc` mesh);
5. results rescaled to several cooling times (`get_scaling_factors_cooling_time`,
   `rescale_dose_map`) and collected in an atlas (`MeshPlotter`, `Atlas`).